# no-grad-context-mgr-update — worked example 1: NoGrad with save-and-restore on exit

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-grad-context-mgr-update`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `NoGrad` context manager disables a module-level `grad_tracking_enabled` flag inside its block. The correct pattern saves the PREVIOUS value on enter and restores it on exit, so nested `with NoGrad()` blocks compose without the inner exit wrongly re-enabling grad inside an outer NoGrad.

## Worked solution

We keep a module-level flag `grad_tracking_enabled` and write `NoGrad` as a class. In `__enter__` we declare the name `global`, stash the current value in `self._prev`, set the flag to `False`, and return `self`. In `__exit__` we restore `self._prev` rather than hard-coding `True`, which is what makes nesting safe. We then demonstrate the flag is `True` outside, `False` inside a single `with NoGrad()`, and back to `True` afterward. We print the flag value at each stage to show the save-restore behavior.

In [ ]:
grad_tracking_enabled = True

class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev

print('before:', grad_tracking_enabled)
with NoGrad():
    print('inside:', grad_tracking_enabled)
print('after:', grad_tracking_enabled)